# Pipeline Adaptive Threshold Raw Data

This notebook runs the full raw-data walk-forward version:

- load long daily and 5-minute data
- train the daily model walk-forward
- train the 5-minute model walk-forward
- generate per-day three-action rewards from the 5-minute engine
- choose daily thresholds online from that history
- trade forward with the chosen gate


In [ ]:
from adaptive_threshold_raw_walkforward import run_raw_adaptive_threshold_walkforward
from adaptive_trade_extensions import RollingThresholdConfig, ThresholdPairBanditConfig, make_threshold_grid

import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# ------------------------------------------------------------
# Main run configuration
# ------------------------------------------------------------

policy_mode = 'adaptive_reward'
# options:
#   'static'
#   'adaptive_reward'
#   'adaptive_accuracy'
#   'threshold_pair_bandit'

run_kwargs = dict(
    daily_csv_path='DataAPI/data/QQQ_DAY.csv',
    k5m_csv_path='DataAPI/data/QQQ_5M.csv',
    code='QQQ',

    daily_chan_start='2014-06-01',
    accumulation_start='2016-10-01',
    sim_start='2019-01-01',
    end_time='2026-12-31',

    N_confirm=5,
    min_labeled_days_to_train=200,
    retrain_every_new_labels=25,
    dp_lookback=5,

    lookahead_days_5m=2.0,
    retrain_every_days_5m=5,
    min_samples_total_5m=300,

    threshold_window_days=2.0,
    threshold_ret_grid=None,
    threshold_min_open_signals=10,

    initial_capital=100000.0,
    fee_pct=0.0,

    daily_chan_max_klines=500,
    five_chan_max_klines=500,

    macro_files={
        'vix_': 'VIX.csv',
        'dxy_': 'DXY.csv',
        'us5y_': 'US5Y.csv',
        'us10y_': 'US10Y.csv',
        'us30y_': 'US30Y.csv',
        'xau_': 'XAU.csv',
        'nyxbt_': 'NYXBT.csv',
        '10y2ys_': '10Y2YS.csv',
        'us2y_': 'US2Y.csv',
        'spgsci_': 'SPGSCI.csv',
        'spy_': 'SPY_DAY.csv',
        'qqq_': 'QQQ_DAY.csv',
        'rut_': 'RUT.csv',
        'vvix_': 'VVIX.csv',
        'vix3m_': 'VIX3M.csv',
        'vxn_': 'VXN.csv',
        'rvx_': 'RVX.csv',
        'ixic_': 'IXIC.csv',
        'ndx_': 'NDX.csv',
        'dji_': 'DJI.csv',
    },

    policy_mode=policy_mode,
    static_buy_level=0.20,
    static_sell_level=0.30,

    daily_threshold_config=RollingThresholdConfig(
        lookback_days=252,
        buy_grid=make_threshold_grid(0.05, 0.35, 0.005),
        sell_grid=make_threshold_grid(0.15, 0.60, 0.005),
        min_gap=0.02,
        min_obs=60,
        switch_penalty=0.0,
    ),

    threshold_pair_bandit_config=ThresholdPairBanditConfig(
        threshold_pairs=[
            (0.10, 0.22),
            (0.12, 0.24),
            (0.15, 0.25),
            (0.18, 0.28),
            (0.20, 0.30),
            (0.22, 0.32),
            (0.25, 0.35),
        ],
        alpha=0.50,
        l2=1.0,
    ),

    output_dir=f'output_raw_adaptive_threshold_{policy_mode}_QQQ',
    verbose=True,
)

run_kwargs

In [ ]:
res = run_raw_adaptive_threshold_walkforward(**run_kwargs)
res['output_dir']

In [ ]:
res['daily_log_df'].tail(20)

In [ ]:
res['daily_reward_df'].tail(20)

In [ ]:
res['trades_df'].tail(20)

In [ ]:
daily_df = res['daily_log_df'].copy()
reward_df = res['daily_reward_df'].copy()

if not daily_df.empty:
    daily_df['date'] = pd.to_datetime(daily_df['date'])

if not reward_df.empty:
    reward_df['date'] = pd.to_datetime(reward_df['date'])

summary = {
    'final_equity': float(daily_df['equity'].iloc[-1]) if not daily_df.empty else None,
    'num_trades': int(len(res['trades_df'])),
    'num_reward_days': int(len(reward_df)),
}
summary

In [ ]:
if not daily_df.empty:
    plt.figure(figsize=(14, 6))
    plt.plot(daily_df['date'], daily_df['equity'], label='strategy equity')
    plt.legend()
    plt.title('Raw Adaptive Threshold Walk-Forward Equity')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
if not daily_df.empty:
    plt.figure(figsize=(14, 6))
    plt.plot(daily_df['date'], daily_df['p_day'], label='p_day')
    plt.plot(daily_df['date'], daily_df['daily_buy_level'], label='daily_buy_level')
    plt.plot(daily_df['date'], daily_df['daily_sell_level'], label='daily_sell_level')
    plt.legend()
    plt.title('Daily Probability and Adaptive Thresholds')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## Resume From Existing Checkpoint

The new raw adaptive-threshold helper does not yet support checkpoint resume.
If you need to continue from an existing saved checkpoint after cleaning duplicated 5-minute data,
use the checkpoint-aware pipeline in `pipelineCurrent.py`.

In [ ]:
from pipelineCurrent import run_daily_bandit_then_5m_xgb

In [ ]:
# ------------------------------------------------------------
# Resume from a previous checkpoint after fixing the 5m data
# ------------------------------------------------------------

resume_kwargs = dict(
    daily_csv_path='DataAPI/data/QQQ_DAY.csv',
    k5m_csv_path='DataAPI/data/QQQ_5M.csv',
    code='QQQ',

    daily_chan_start='2014-06-01',
    accumulation_start='2016-10-01',
    sim_start='2019-01-01',
    end_time='2026-12-31',

    N_confirm=5,
    min_labeled_days_to_train=200,
    retrain_every_new_labels=25,
    dp_lookback=5,

    bandit_alpha=0.75,
    bandit_l2=1.0,

    lookahead_days_5m=2.0,
    retrain_every_days_5m=5,
    min_samples_total_5m=300,

    threshold_window_days=2.0,
    threshold_ret_grid=None,
    threshold_min_open_signals=10,

    initial_capital=100000.0,
    fee_pct=0.0,

    daily_chan_max_klines=500,
    five_chan_max_klines=500,

    macro_files={
        'vix_': 'VIX.csv',
        'dxy_': 'DXY.csv',
        'us5y_': 'US5Y.csv',
        'us10y_': 'US10Y.csv',
        'us30y_': 'US30Y.csv',
        'xau_': 'XAU.csv',
        'nyxbt_': 'NYXBT.csv',
        '10y2ys_': '10Y2YS.csv',
        'us2y_': 'US2Y.csv',
        'spgsci_': 'SPGSCI.csv',
        'spy_': 'SPY_DAY.csv',
        'qqq_': 'QQQ_DAY.csv',
        'rut_': 'RUT.csv',
        'vvix_': 'VVIX.csv',
        'vix3m_': 'VIX3M.csv',
        'vxn_': 'VXN.csv',
        'rvx_': 'RVX.csv',
        'ixic_': 'IXIC.csv',
        'ndx_': 'NDX.csv',
        'dji_': 'DJI.csv',
    },

    output_dir='output_resumed_from_checkpoint_after_dedup_QQQ',
    verbose=True,

    save_checkpoint_path='checkpoints/QQQ_daily_bandit_5m_checkpoint.joblib',
    checkpoint_every_n_days=5,
    resume_from_checkpoint_path='checkpoints/QQQ_daily_bandit_5m_checkpoint.joblib',
)

resume_kwargs

In [ ]:
# Uncomment to continue from the last checkpoint using the cleaned data
# resumed = run_daily_bandit_then_5m_xgb(**resume_kwargs)
# resumed['output_dir']